# Small-Scale Gaussian Splatting from a Drone Video

This notebook runs in a fresh Google Colab runtime or in the configured local ROCm environment.

Pipeline:
1. Install dependencies
2. Upload/verify the drone video
3. Extract overlapping frames with FFmpeg
4. Run COLMAP feature extraction, matching, and incremental mapping
5. Inspect the recovered reconstruction
6. Create the gsplat COLMAP dataset
7. Clone the gsplat source repository
8. Train a small Gaussian-splat model and save a `.ply`


## 1. Install Dependencies


In [1]:
# Install Colab dependencies, or use the local venv for preprocessing.
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg", "git"], check=True)

    # Keep Colab's CUDA-enabled PyTorch and torchvision versions.
    subprocess.run([
        sys.executable, "-m", "pip", "install",
        "numpy>=2,<3", "pillow", "matplotlib", "tqdm", "imageio[ffmpeg]",
        "pycolmap>=3.10", "scikit-learn", "torchmetrics==1.8.2",
        "tyro>=0.8.8,!=1.0.9,!=1.0.10", "viser", "tensorboard", "tensorly",
        "pyyaml", "ninja", "jaxtyping", "rich>=12", "typing_extensions",
        "nvidia-ncore>=19.0.0", "piexif", "splines",
    ], check=True)

    GSPLAT_REPO = Path("/content/gsplat")
    if GSPLAT_REPO.exists():
        shutil.rmtree(GSPLAT_REPO)
    subprocess.run([
        "git", "clone", "--recurse-submodules", "--depth", "1",
        "--branch", "main", "https://github.com/nerfstudio-project/gsplat.git",
        str(GSPLAT_REPO),
    ], check=True)

    # Defer CUDA extension compilation until first use.
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--no-deps",
        "--no-build-isolation", "-e", str(GSPLAT_REPO),
    ], env={**os.environ, "BUILD_NO_CUDA": "1", "MAX_JOBS": "2"}, check=True)
else:
    GSPLAT_REPO = Path.cwd() / "gsplat"
    rocm_torch_lib = Path(sys.executable).parent.parent / "lib/python3.12/site-packages/torch/lib"
    if rocm_torch_lib.exists():
        os.environ.setdefault("HSA_ENABLE_DXG_DETECTION", "1")
        os.environ["LD_LIBRARY_PATH"] = f"{rocm_torch_lib}:/opt/rocm/lib" + (
            f":{os.environ['LD_LIBRARY_PATH']}" if os.environ.get("LD_LIBRARY_PATH") else ""
        )
    required = ["cv2", "numpy", "matplotlib", "tqdm", "imageio_ffmpeg", "pycolmap"]
    missing = [name for name in required if importlib.util.find_spec(name) is None]
    if missing:
        raise RuntimeError(
            f"Missing local packages: {missing}. Run "
            "./run-rocm.sh, then select the Gaussian Splat ROCm kernel."
        )
    print("Local preprocessing environment detected.")
    print("Gaussian training uses the configured CUDA or ROCm gsplat build.")


Local preprocessing environment detected.
Gaussian training uses the configured CUDA or ROCm gsplat build.


## 2. Imports


In [2]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import pycolmap

print("Python:", sys.version)


Python: 3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]


## 3. User Configuration


In [3]:
# =========================
# User configuration
# =========================

# The video and outputs live beside this notebook in both Colab and local runs.
VIDEO_PATH = Path.cwd() / "drone-review.mp4"

# Extraction:
VIDEO_FPS = 24.0          # 6 fps gives useful overlap while reducing redundancy
MAX_FRAMES = None         # None = use the entire video
IMAGE_WIDTH = None      # None = keep original width
IMAGE_HEIGHT = None      # None = preserve aspect ratio

# Output:
OUTPUT_DIR = Path.cwd() / "gaussian_splat"

# COLMAP:
CAMERA_MODEL = "OPENCV"

# gsplat:
NUM_ITERATIONS = 40000
SH_DEGREE = 4

# Preview:
SHOW_PREVIEW = True

print("Video:", VIDEO_PATH)
print("Output:", OUTPUT_DIR)


Video: /home/bren/Guass/drone-review.mp4
Output: /home/bren/Guass/gaussian_splat


## 4. Verify / Upload Video


In [4]:
# In Colab, prompt for upload. Locally, put the video beside this notebook.
if not VIDEO_PATH.exists():
    try:
        from google.colab import files
        print(f"{VIDEO_PATH.name} was not found. Please upload it.")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No video was uploaded.")
        uploaded_name = VIDEO_PATH.name if VIDEO_PATH.name in uploaded else next(iter(uploaded))
        uploaded_path = Path("/content") / uploaded_name
        VIDEO_PATH.parent.mkdir(parents=True, exist_ok=True)
        if uploaded_path != VIDEO_PATH:
            shutil.copy2(uploaded_path, VIDEO_PATH)
    except ImportError:
        raise FileNotFoundError(
            f"{VIDEO_PATH} was not found. Place the video beside this notebook "
            "or change VIDEO_PATH in the configuration cell."
        )

if not VIDEO_PATH.exists():
    raise FileNotFoundError(f"Video not found: {VIDEO_PATH}")

print("Video found:", VIDEO_PATH)
print("Size:", f"{VIDEO_PATH.stat().st_size / 1e6:.1f} MB")


Video found: /home/bren/Guass/drone-review.mp4
Size: 930.4 MB


## 5. Verify Video


In [5]:
video = cv2.VideoCapture(str(VIDEO_PATH))

if not video.isOpened():
    raise RuntimeError(f"OpenCV could not open video: {VIDEO_PATH}")

fps = video.get(cv2.CAP_PROP_FPS)
frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

video.release()

if fps <= 0 or frame_count <= 0 or width <= 0 or height <= 0:
    raise RuntimeError(
        "Video metadata is invalid. The file may be corrupt or use an unsupported codec."
    )

duration = frame_count / fps

print(f"Source FPS:   {fps:.2f}")
print(f"Frames:       {frame_count}")
print(f"Resolution:   {width} x {height}")
print(f"Duration:     {duration:.2f} sec")

if VIDEO_FPS is not None and VIDEO_FPS > fps:
    print(
        f"Warning: requested {VIDEO_FPS:.2f} FPS exceeds source FPS. "
        f"Using every available frame instead."
    )


Source FPS:   12.00
Frames:       734
Resolution:   3840 x 2160
Duration:     61.17 sec


## 6. Robust Frame Extraction


In [ ]:
def _clear_jpgs(directory):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    for p in directory.glob("*.jpg"):
        p.unlink()


def extract_frames_ffmpeg(
    video_path,
    output_dir,
    target_fps=None,
    max_frames=None,
    output_width=None,
    output_height=None,
):
    """Extract frames with FFmpeg and verify that files were actually created."""

    video_path = Path(video_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    _clear_jpgs(output_dir)

    # If the frame cap is active, lower the extraction rate so samples span
    # the whole video instead of taking only its opening segment.
    if max_frames is not None:
        capture = cv2.VideoCapture(str(video_path))
        source_fps = capture.get(cv2.CAP_PROP_FPS)
        source_count = capture.get(cv2.CAP_PROP_FRAME_COUNT)
        capture.release()
        if source_fps > 0 and source_count > 0:
            duration = source_count / source_fps
            cap_fps = max_frames / duration
            target_fps = cap_fps if target_fps is None else min(target_fps, cap_fps)

    # Build the video filter.
    filters = []

    if target_fps is not None:
        filters.append(f"fps={float(target_fps):g}")

    if output_width is not None:
        if output_height is None:
            filters.append(
                f"scale={int(output_width)}:-2:flags=lanczos"
            )
        else:
            filters.append(
                f"scale={int(output_width)}:{int(output_height)}:flags=lanczos"
            )

    ffmpeg_bin = shutil.which("ffmpeg")
    if ffmpeg_bin is None:
        import imageio_ffmpeg
        ffmpeg_bin = imageio_ffmpeg.get_ffmpeg_exe()

    cmd = [
        ffmpeg_bin,
        "-hide_banner",
        "-loglevel", "error",
        "-y",
        "-i", str(video_path),
    ]

    if filters:
        cmd += ["-vf", ",".join(filters)]

    if max_frames is not None:
        cmd += ["-frames:v", str(int(max_frames))]

    cmd += [
        "-q:v", "2",
        str(output_dir / "%06d.jpg"),
    ]

    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    images = sorted(output_dir.glob("*.jpg"))

    if result.returncode != 0:
        raise RuntimeError(
            "FFmpeg frame extraction failed.\n\n"

            + result.stderr[-4000:]
        )

    if not images:
        raise RuntimeError(
            "FFmpeg completed but produced zero images.\n"
            f"Video: {video_path}\n"
            f"Output directory: {output_dir}"
        )

    return images


def extract_frames_opencv(
    video_path,
    output_dir,
    target_fps=None,
    max_frames=None,
    output_width=None,
    output_height=None,
):
    """Fallback extractor with explicit read/write checks."""

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    _clear_jpgs(output_dir)

    video = cv2.VideoCapture(str(video_path))

    if not video.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    source_fps = video.get(cv2.CAP_PROP_FPS)

    if source_fps <= 0:
        video.release()
        raise RuntimeError("Invalid source FPS.")

    source_frame_count = video.get(cv2.CAP_PROP_FRAME_COUNT)
    if max_frames is not None and source_frame_count > 0:
        duration = source_frame_count / source_fps
        cap_fps = max_frames / duration
        target_fps = cap_fps if target_fps is None else min(target_fps, cap_fps)

    if target_fps is None or target_fps >= source_fps:
        frame_step = 1
    else:
        frame_step = max(1, int(round(source_fps / target_fps)))

    extracted = 0
    frame_idx = 0

    while True:
        ret, frame = video.read()

        if not ret:
            break

        if frame_idx % frame_step == 0:
            h, w = frame.shape[:2]

            if output_width is not None:
                if output_height is None:
                    scale = output_width / w
                    new_w = int(output_width)
                    new_h = max(1, int(round(h * scale)))
                else:
                    new_w = int(output_width)
                    new_h = int(output_height)

                frame = cv2.resize(
                    frame,
                    (new_w, new_h),
                    interpolation=cv2.INTER_AREA,
                )

            filename = output_dir / f"{extracted:06d}.jpg"

            ok = cv2.imwrite(
                str(filename),
                frame,
                [cv2.IMWRITE_JPEG_QUALITY, 95],
            )

            if not ok or not filename.exists() or filename.stat().st_size == 0:
                video.release()
                raise RuntimeError(f"Failed to write frame: {filename}")

            extracted += 1

            if max_frames is not None and extracted >= max_frames:
                break

        frame_idx += 1

    video.release()

    images = sorted(output_dir.glob("*.jpg"))

    if not images:
        raise RuntimeError(
            "OpenCV completed without producing any images."
        )

    return images


def extract_frames(
    video_path,
    output_dir,
    target_fps=None,
    max_frames=None,
    output_width=None,
    output_height=None,
):
    """FFmpeg-first extraction with a verified OpenCV fallback."""

    try:
        images = extract_frames_ffmpeg(
            video_path,
            output_dir,
            target_fps,
            max_frames,
            output_width,
            output_height,
        )
        print("Frame extraction method: FFmpeg")
    except Exception as ffmpeg_error:
        print("FFmpeg extraction failed; trying OpenCV fallback.")
        print(str(ffmpeg_error))

        images = extract_frames_opencv(
            video_path,
            output_dir,
            target_fps,
            max_frames,
            output_width,
            output_height,
        )
        print("Frame extraction method: OpenCV fallback")

    # Verify that the images are readable, not merely present.
    valid_images = []
    for image_path in images:
        image = cv2.imread(str(image_path))
        if image is not None and image.size > 0:
            valid_images.append(image_path)

    if len(valid_images) < 5:
        raise RuntimeError(
            f"Only {len(valid_images)} valid images were extracted. "
            "COLMAP needs substantially more useful views."
        )

    print(f"Extracted {len(valid_images)} valid images")
    print(f"First image: {valid_images[0].name}")
    print(f"Last image:  {valid_images[-1].name}")

    return valid_images


image_dir = OUTPUT_DIR / "images"

images = extract_frames(
    VIDEO_PATH,
    image_dir,
    target_fps=VIDEO_FPS,
    max_frames=MAX_FRAMES,
    output_width=IMAGE_WIDTH,
    output_height=IMAGE_HEIGHT,
)


Frame extraction method: FFmpeg


## 7. Preview Frames


In [ ]:
# Preview several extracted frames.
if SHOW_PREVIEW:
    n_preview = min(6, len(images))
    indices = np.linspace(0, len(images) - 1, n_preview, dtype=int)

    fig, axes = plt.subplots(
        2,
        3,
        figsize=(15, 8),
    )
    axes = np.asarray(axes).ravel()

    for ax in axes:
        ax.axis("off")

    for ax, idx in zip(axes, indices):
        img = cv2.imread(str(images[idx]))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f"Frame {idx}")

    plt.tight_layout()
    plt.show()


## 8. COLMAP Setup


In [ ]:
# Start from a clean COLMAP workspace.
colmap_dir = OUTPUT_DIR / "colmap"
database_path = colmap_dir / "database.db"
sparse_dir = colmap_dir / "sparse"

if colmap_dir.exists():
    shutil.rmtree(colmap_dir)

colmap_dir.mkdir(parents=True, exist_ok=True)
sparse_dir.mkdir(parents=True, exist_ok=True)

print("COLMAP database:", database_path)
print("Image directory:", image_dir)
print("Camera model:", CAMERA_MODEL)


## 9. COLMAP Feature Extraction


In [ ]:
# Configure one shared camera model for all frames from the same video camera.
pycolmap.logging.minloglevel = pycolmap.logging.Level.WARNING
reader_options = pycolmap.ImageReaderOptions()
reader_options.camera_model = CAMERA_MODEL

pycolmap.extract_features(
    database_path=str(database_path),
    image_path=str(image_dir),
    camera_mode=pycolmap.CameraMode.SINGLE,
    reader_options=reader_options,
)

print("Feature extraction complete.")


## 10. COLMAP Feature Matching


In [ ]:
# Match a wider temporal neighborhood for reliable video reconstruction.
pairing_options = pycolmap.SequentialPairingOptions()
pairing_options.overlap = 20
pycolmap.match_sequential(
    database_path=str(database_path),
    pairing_options=pairing_options,
)

print("Sequential feature matching complete.")


## 11. COLMAP Reconstruction


In [ ]:
# Incremental reconstruction with deterministic initialization and fixed lens distortion.
mapping_options = pycolmap.IncrementalPipelineOptions()
mapping_options.random_seed = 0
mapping_options.mapper.random_seed = 0
mapping_options.ba_refine_extra_params = False
mapping_options.mapper.abs_pose_refine_extra_params = False
reconstruction = pycolmap.incremental_mapping(
    database_path=str(database_path),
    image_path=str(image_dir),
    output_path=str(sparse_dir),
    options=mapping_options,
)

print("COLMAP reconstruction complete.")
print("Reconstructions:", len(reconstruction))

if len(reconstruction) == 0:
    raise RuntimeError(
        "COLMAP did not produce a reconstruction. "
        "Try increasing MAX_FRAMES, reducing VIDEO_FPS, "
        "or using a video segment with stronger camera motion and texture."
    )


## 12. Validate Reconstruction


In [ ]:
# Select the reconstruction with the most registered images.
recon = max(
    reconstruction.values(),
    key=lambda r: len(r.images),
)

registered_images = len(recon.images)
num_points = len(recon.points3D)

print("Registered images:", registered_images)
print("3D points:", num_points)

if registered_images < 5:
    raise RuntimeError(
        f"Only {registered_images} images were registered by COLMAP. "
        "This is too few for a useful Gaussian-splat reconstruction."
    )

if num_points < 100:
    print(
        "Warning: COLMAP recovered fewer than 100 3D points. "
        "The reconstruction may be too sparse for good splatting."
    )


## 13. Inspect Point Cloud


In [ ]:
# Inspect the recovered point cloud.
points = np.array([
    point.xyz
    for point in recon.points3D.values()
])

colors = np.array([
    point.color
    for point in recon.points3D.values()
]) / 255.0

print("Point cloud shape:", points.shape)

if len(points) > 0:
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")

    ax.scatter(
        points[:, 0],
        points[:, 1],
        points[:, 2],
        c=colors,
        s=1,
        alpha=0.5,
    )

    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.set_title("COLMAP Sparse Point Cloud")

    plt.show()


## 14. Inspect Camera Poses


In [ ]:
# Recover camera centers.
camera_centers = []
camera_rotations = []
camera_names = []

for image_id, image in recon.images.items():
    cam_from_world = image.cam_from_world()

    R = cam_from_world.rotation.matrix()
    t = np.asarray(cam_from_world.translation)

    C = -R.T @ t

    camera_centers.append(C)
    camera_rotations.append(R)
    camera_names.append(image.name)

camera_centers = np.asarray(camera_centers)
camera_rotations = np.asarray(camera_rotations)

print("Camera centers:", camera_centers.shape)
print("Camera rotations:", camera_rotations.shape)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

if len(points):
    ax.scatter(
        points[:, 0],
        points[:, 1],
        points[:, 2],
        c=colors,
        s=1,
        alpha=0.4,
    )

ax.scatter(
    camera_centers[:, 0],
    camera_centers[:, 1],
    camera_centers[:, 2],
    s=25,
    marker="^",
)

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_title("COLMAP Reconstruction + Camera Positions")

plt.show()


## 15. Build gsplat Dataset


In [ ]:
# Create the COLMAP dataset layout expected by the gsplat examples.
GSPLAT_DATA = OUTPUT_DIR / "colmap_scene"
GSPLAT_IMAGE_DIR = GSPLAT_DATA / "images"
GSPLAT_IMAGE_DIR_4 = GSPLAT_DATA / "images_4"
GSPLAT_SPARSE_DIR = GSPLAT_DATA / "sparse" / "0"

if GSPLAT_DATA.exists():
    shutil.rmtree(GSPLAT_DATA)

GSPLAT_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
GSPLAT_IMAGE_DIR_4.mkdir(parents=True, exist_ok=True)
GSPLAT_SPARSE_DIR.mkdir(parents=True, exist_ok=True)

# Write the selected reconstruction.
recon.write(GSPLAT_SPARSE_DIR)

# Copy all extracted images. COLMAP image names remain unchanged.
for image_file in images:
    shutil.copy2(
        image_file,
        GSPLAT_IMAGE_DIR / image_file.name,
    )
    source_image = cv2.imread(str(image_file))
    if source_image is None:
        raise RuntimeError(f"Could not read {image_file} for gsplat downscaling.")
    height, width = source_image.shape[:2]
    downscaled = cv2.resize(source_image, (width // 4, height // 4), interpolation=cv2.INTER_AREA)
    if not cv2.imwrite(str(GSPLAT_IMAGE_DIR_4 / image_file.name), downscaled):
        raise RuntimeError(f"Could not write downscaled image {image_file.name}.")

print("gsplat dataset created:")
print(GSPLAT_DATA)
print("Images:", len(list(GSPLAT_IMAGE_DIR.glob("*.jpg"))))
print("Training images (1/4 scale):", len(list(GSPLAT_IMAGE_DIR_4.glob("*.jpg"))))
print("Sparse files:")
for p in sorted(GSPLAT_SPARSE_DIR.iterdir()):
    print("  ", p.name)


## 16. Verify GPU (CUDA or ROCm)


In [ ]:
try:
    import torch
except ImportError as exc:
    raise RuntimeError(
        "PyTorch with GPU support is not installed in this local environment. "
        "For AMD WSL training, set up ROCm PyTorch and the ROCm gsplat port first."
    ) from exc

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("ROCm/HIP version:", torch.version.hip)

if not torch.cuda.is_available():
    raise RuntimeError(
        "No supported GPU runtime is available. Install a CUDA build for NVIDIA, "
        "or ROCm PyTorch plus a ROCm-compatible gsplat build for AMD."
    )

print("GPU:", torch.cuda.get_device_name(0))


## 17. Prepare gsplat Source


In [ ]:
# The dependency cell prepares this path in Colab. Local AMD setup uses a ROCm gsplat checkout here.
trainer = GSPLAT_REPO / "examples" / "simple_trainer.py"

if not trainer.exists():
    raise FileNotFoundError(
        f"gsplat trainer was not found at {trainer}. "
        "For local AMD training, install a ROCm-compatible gsplat checkout first."
    )

print("gsplat source:", GSPLAT_REPO)
print("Trainer:", trainer)


## 18. Verify gsplat Trainer


In [ ]:
# Verify that the trainer can see the dataset before spending time training.
help_result = subprocess.run(
    [
        sys.executable,
        str(trainer),
        "--help",
    ],
    cwd=str(GSPLAT_REPO),
    env={**os.environ, "PYTHONPATH": str(GSPLAT_REPO)},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(help_result.stdout[-8000:])

if help_result.returncode != 0:
    raise RuntimeError("gsplat simple_trainer.py --help failed.")


## 19. Train Gaussian Splat


In [ ]:
# Train the Gaussian-splat model.
#
# The output is written under /content/gaussian_splat/trained.
# This is intentionally a small 2k-step run for the small-scale experiment.

TRAIN_DIR = OUTPUT_DIR / "trained"
TRAIN_DIR.mkdir(parents=True, exist_ok=True)

train_cmd = [
    sys.executable,
    str(trainer),
    "default",
    "--data_dir", str(GSPLAT_DATA),
    "--result_dir", str(TRAIN_DIR),
    "--max_steps", str(NUM_ITERATIONS),
    "--init_type", "sfm",
    "--sh_degree", str(SH_DEGREE),
    "--disable_viewer",
    "--save_ply",
    "--ply_steps", str(NUM_ITERATIONS),
]

print("Running:")
print(" ".join(train_cmd))

result = subprocess.run(
    train_cmd,
    cwd=str(GSPLAT_REPO),
    env={**os.environ, "PYTHONPATH": str(GSPLAT_REPO)},
)

if result.returncode != 0:
    raise RuntimeError(
        f"gsplat training failed with exit code {result.returncode}."
    )

ply_files = sorted(TRAIN_DIR.rglob("*.ply"))

if not ply_files:
    raise RuntimeError(
        f"Training completed but no .ply file was found under {TRAIN_DIR}."
    )

print("\nTraining complete.")
print("PLY files:")
for p in ply_files:
    print("  ", p)


## 20. Final Summary


In [ ]:
# Final summary.
print("=" * 60)
print("DONE")
print("=" * 60)
print("Input video:       ", VIDEO_PATH)
print("Extracted images:  ", len(images))
print("Registered images: ", registered_images)
print("COLMAP points:     ", num_points)
print("Dataset:            ", GSPLAT_DATA)
print("Training output:    ", TRAIN_DIR)
print("PLY files:")

for p in sorted(TRAIN_DIR.rglob("*.ply")):
    print("  ", p)
